In [14]:
import gradio as gr
import numpy as np
from PIL import Image
from sklearn.datasets import fetch_lfw_people
import matplotlib.pyplot as plt
import numpy as np
import matplotlib
matplotlib.use('TkAgg')

def redimensionare(A,l,c):
    m,n=np.shape(A)
    puncte_linii=np.linspace(0,m,l+1).astype(int)
    puncte_coloane = np.linspace(0, n, c+1).astype(int)
    B=np.zeros((l,c))
    for i in range (l):
        for j in range(c):
            bloc=A[puncte_linii[i]:puncte_linii[i+1],puncte_coloane[j]:puncte_coloane[j+1]]
            B[i,j]=np.sum(bloc)/((puncte_linii[i+1]-puncte_linii[i])*(puncte_coloane[j+1]-puncte_coloane[j]))
    return B


def Imagini():   
    lfw_people = fetch_lfw_people(min_faces_per_person=20, color=False, resize=1.0)
    n_samples, h, w = lfw_people.images.shape
    return lfw_people.images, n_samples, h, w

def centrare_date(A):
    fata_medie = np.mean(X, axis=1).reshape(-1, 1)
    A_centrat = A - fata_medie
    return A_centrat, fata_medie.flatten()
    """
    -- Functia mea (logica corecta, dar este mult mai lenta) --
    m,n=np.shape(A)
    fata_medie=[]
    for i in range (m):
        s=0
        for j in range (n):
           s+=A[i][j]
        s=s/n
        fata_medie.append(s)
    Fata_medie=np.array(fata_medie).T
    A_centrat = A.copy().astype(float)
    for i in range (n):
        A_centrat[:, i] = A_centrat[:, i] - Fata_medie    
    return A_centrat, Fata_medie"""

def Tridiag_Householder(A):
    n = np.shape(A)[0]
    T = np.copy(A)
    Q = np.eye(n)

    for k in range(n - 2):
        v = np.copy(T[k + 1:, k]).reshape(-1,1)
        norma_v = np.linalg.norm(v)
        s = 1 if v[0] >= 0 else -1
        v[0] += s * norma_v
        Hv=np.eye(n-k-1)-2*(v@v.T)/(v.T@v)
        H = np.block([
                [np.eye(k + 1),np.zeros((k + 1, n - k - 1))],
                [np.zeros((n - k - 1, k + 1)), Hv]
            ])
        T=H@T@H
        Q=Q@H
    return Q, T

def QR(A):
    m, n = A.shape
    Q = np.eye(m)
    R = A.copy().astype(float)
    
    for k in range(n):
        v = np.copy(R[k:, k]).reshape(-1, 1)
        norm_v = np.linalg.norm(v)
        
        # Dacă coloana e deja zero, sărim peste ea
        if norm_v < 1e-15:
            continue
            
        s = 1 if v[0] >= 0 else -1
        v[0] += s * norm_v
        
        # Recalculăm numitorul și verificăm din nou
        numitor = v.T @ v
        if numitor > 1e-15:
            Hv = np.eye(m - k) - 2 * (v @ v.T) / numitor
            H = np.eye(m)
            H[k:, k:] = Hv
            
            R = H @ R
            Q = Q @ H
            
    return Q, R

def QR_iteration(Mat_L, Q_tri, TOL=1e-2, max_iter=100):
    T = Q_tri.T @ Mat_L @ Q_tri 
    V = Q_tri.copy()
    
    # Înlocuim bucla infinită while cu un for controlat
    for _ in range(max_iter):        
        # Verificăm dacă matricea e suficient de diagonală
        if np.sum(np.abs(T - np.diag(np.diag(T)))) < TOL:
            break
            
        Q_k, R_k = QR(T) 
        T = R_k @ Q_k
        V = V @ Q_k

    n_local = T.shape[0]
    for i in range(n_local - 1):
        idx_maxim = i + np.argmax(np.diag(T)[i:])
        if idx_maxim != i:
            T[[i, idx_maxim], :] = T[[idx_maxim, i], :]
            T[:, [i, idx_maxim]] = T[:, [idx_maxim, i]]
            V[:, [i, idx_maxim]] = V[:, [idx_maxim, i]]
            
    return T, V

def calculeaza_sosia_dupa_selectie(imagine_de_la_interfata,k_componente):
    if imagine_de_la_interfata is None:
        return None
    
    img = Image.fromarray(imagine_de_la_interfata).convert('L')
    IMG = redimensionare(np.array(img), linii, coloane)
    
    v_tu = IMG.flatten().astype(np.float64)
    v_tu_normat = v_tu / np.linalg.norm(v_tu)
    v_tu_centrat = v_tu_normat - Fata_medie
    
    k = int(k_componente)
    U_redus = U[:, :k]
    W_redus = W[:k, :]
    
    w_tu = U_redus.T @ v_tu_centrat
    distante = np.linalg.norm(W_redus - w_tu[:, np.newaxis], axis=0)
    index_minim = np.argmin(distante)
    
    sosia_vector = X[:, index_minim]
    sosia_matrice = sosia_vector.reshape(linii, coloane)
    sosia_0_255 = ((sosia_matrice - sosia_matrice.min()) / (sosia_matrice.max() - sosia_matrice.min()) * 255).astype(np.uint8)
    
    return sosia_0_255

Fete, n, linii, coloane=Imagini()

ok=0
X_list=[]
for i in range (101):
    if linii>200 or coloane>200:
        img_prelucrata=redimensionare(Fete[i],200,200)
        ok=1
    else: img_prelucrata=Fete[i]
    v_brut = img_prelucrata.flatten().astype(np.float64)
    v_normat = v_brut / np.linalg.norm(v_brut) # Norma aici!
    X_list.append(v_normat)
X = np.array(X_list).T
if ok==1:
    linii,coloane=200,200
A,Fata_medie=centrare_date(X)

"""
plt.imshow(Fata_medie.reshape(linii, coloane), cmap='gray')
plt.show()
"""

L=A.T@A
Q_tri,T=Tridiag_Householder(L)
T_final, V_final=QR_iteration(L,Q_tri)
U = A @ V_final 

for i in range(U.shape[1]):
    U[:, i] = U[:, i] / np.linalg.norm(U[:, i])

"""
Afișăm prima Eigenface (cea mai importantă)
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.imshow(U[:, 0].reshape(linii, coloane), cmap='gray')
plt.title("Eigenface 1 (Cea mai mare valoare proprie)")

Afișăm a doua Eigenface (următoarea ca importanță)
plt.subplot(1, 2, 2)
plt.imshow(U[:, 1].reshape(linii, coloane), cmap='gray')
plt.title("Eigenface 2")
plt.show()
"""
W = U.T @ A
"""
# 1. Extragem ponderile primei fețe (prima coloană din W)
w1 = W[:, 0] 
# 2. Reconstruim fața în spațiul pixelilor
# Înmulțim matricea U (Eigenfaces) cu vectorul de ponderi w1
fata_reconstruita_centrata =  U[:, :40] @ w1
# 3. Adăugăm înapoi Fața Medie (psi) pentru a reveni la aspectul original
# (Presupunând că variabila ta pentru fața medie se numește 'psi' sau 'mean_face')
fata_finala = fata_reconstruita_centrata + Fata_medie.flatten()
# 4. Afișăm rezultatul
plt.imshow(fata_finala.reshape(linii, coloane), cmap='gray')
plt.title("Prima față reconstruită din ponderile W")
plt.show() 
"""
"""
plt.imshow(img, cmap='gray') 
plt.title("Poza mea pentru Eigenfaces")
plt.axis('off') # Opțional: ascunde axele (numerele de pe margini)
plt.show()
"""

"""plt.imshow(IMG, cmap='gray') 
plt.title("Poza mea pentru Eigenfaces dupa redimensionare")
plt.axis('off') # Opțional: ascunde axele (numerele de pe margini)
plt.show()
"""

demo = gr.Interface(
    fn=calculeaza_sosia_dupa_selectie, 
    inputs=[
        gr.Image(label="1. Alege/Trage poza ta aici"),
        gr.Slider(minimum=1, maximum=40, value=15, step=1, label="Număr Componente Principale (k)")
    ], 
    outputs=gr.Image(label="2. Sosia ta calculată", image_mode="L"),
    title="Recunoaștere Facială Interactivă (Eigenfaces)",
    description="Reglează parametrul k pentru a modifica numărul de caracteristici vectoriale folosite la comparare."
)

demo.launch()







* Running on local URL:  http://127.0.0.1:7873
* To create a public link, set `share=True` in `launch()`.
